In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install wandb

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

nu = 0.01

In [ ]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ghosalsohom2003 (ghosalsohom2003-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
x = torch.linspace(0, 1, 128)
y = torch.linspace(0, 1, 128)
X_grid, Y_grid = torch.meshgrid(x, y, indexing='ij')
coords = torch.stack([X_grid.flatten(), Y_grid.flatten()], dim=1)
coords = 2.0 * coords - 1.0

In [ ]:
class DeepONetDataset(Dataset):
    def __init__(self, X_data, Y_data, coords, n_points=1000):
        self.X_data = torch.tensor(X_data, dtype=torch.float32)
        self.Y_data = torch.tensor(Y_data, dtype=torch.float32)
        self.coords = coords
        self.n_points = n_points

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):

        branch_input = self.X_data[idx]

        indices = torch.randint(0, 128*128, (self.n_points,))
        trunk_input = self.coords[indices]

        target_field = self.Y_data[idx].reshape(3, -1).permute(1,0)
        target = target_field[indices]

        return branch_input, trunk_input, target

# Model

In [ ]:
class BranchNet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(64, latent_dim)

    def forward(self, u):
        features = self.encoder(u).squeeze(-1).squeeze(-1)
        return self.fc(features)


class TrunkNet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, latent_dim * 3)
        )
        self.latent_dim = latent_dim

    def forward(self, x):
        out = self.net(x)
        return out.view(-1, 3, self.latent_dim)


class DeepONet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.branch = BranchNet(latent_dim)
        self.trunk = TrunkNet(latent_dim)
        self.latent_dim = latent_dim

    def forward(self, u, x):

        B, n_pts, _ = x.shape

        branch_out = self.branch(u)
        trunk_out = self.trunk(x.view(-1, 2))
        trunk_out = trunk_out.view(B, n_pts, 3, self.latent_dim)

        branch_out = branch_out.unsqueeze(1).unsqueeze(2)

        output = torch.sum(branch_out * trunk_out, dim=-1)
        return output

# Physics Loss

In [ ]:
def physics_loss(model, branch_input, trunk_input):

    trunk_input.requires_grad_(True)

    pred = model(branch_input, trunk_input)

    u = pred[:,:,0]
    v = pred[:,:,1]
    p = pred[:,:,2]

    grads = torch.ones_like(u)

    u_x = torch.autograd.grad(u, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,0]
    u_y = torch.autograd.grad(u, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,1]

    v_x = torch.autograd.grad(v, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,0]
    v_y = torch.autograd.grad(v, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,1]

    p_x = torch.autograd.grad(p, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,0]
    p_y = torch.autograd.grad(p, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,1]

    u_xx = torch.autograd.grad(u_x, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,0]
    u_yy = torch.autograd.grad(u_y, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,1]

    v_xx = torch.autograd.grad(v_x, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,0]
    v_yy = torch.autograd.grad(v_y, trunk_input, grads, retain_graph=True, create_graph=True)[0][:,:,1]

    continuity = u_x + v_y

    momentum_x = u*u_x + v*u_y + p_x - nu*(u_xx + u_yy)
    momentum_y = u*v_x + v*v_y + p_y - nu*(v_xx + v_yy)

    loss = (
        continuity.pow(2).mean() +
        momentum_x.pow(2).mean() +
        momentum_y.pow(2).mean()
    )

    return loss

In [ ]:
def relative_l2(pred, target):
    num = torch.norm(pred - target, dim=(1,2))
    den = torch.norm(target, dim=(1,2))
    return (num / (den + 1e-8)).mean()

# Training

In [ ]:
data_path = "/content/drive/MyDrive/LDC_data"
geometries = ["harmonics", "nurbs", "skelneton"]

for geometry in geometries:

    print(f"\n==============================")
    print(f"Training PI-DeepONet for: {geometry}")
    print(f"==============================")

    wandb.init(
        project="PI_DeepONet_LDC_uvp",
        name=f"PI_DeepONet_{geometry}",
        reinit=True,
        config={
            "epochs": 100,
            "batch_size": 8,
            "lr": 1e-3,
            "latent_dim": 256,
            "n_points": 1000,
            "lambda_physics": 0.1
        }
    )

    config = wandb.config


    X_data = np.load(
        os.path.join(data_path, f"{geometry}_lid_driven_cavity_X.npz")
    )['data'].astype(np.float32)

    Y_data = np.load(
        os.path.join(data_path, f"{geometry}_lid_driven_cavity_Y.npz")
    )['data'][:, 0:3].astype(np.float32)

    X_mean, X_std = X_data.mean(), X_data.std()
    Y_mean, Y_std = Y_data.mean(), Y_data.std()

    X_data = (X_data - X_mean) / (X_std + 1e-8)
    Y_data = (Y_data - Y_mean) / (Y_std + 1e-8)

    dataset = DeepONetDataset(
        X_data,
        Y_data,
        coords,
        n_points=config.n_points
    )

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size
    )

    model = DeepONet(config.latent_dim).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.lr
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config.epochs
    )


    for epoch in range(config.epochs):


        model.train()

        train_total_loss = 0
        train_data_loss = 0
        train_phys_loss = 0

        for branch_input, trunk_input, target in train_loader:

            branch_input = branch_input.to(device)
            trunk_input = trunk_input.to(device)
            target = target.to(device)

            optimizer.zero_grad()

            pred = model(branch_input, trunk_input)

            # Data Loss
            data_loss = F.mse_loss(pred, target)

            # Physics Loss
            phys_loss = physics_loss(model, branch_input, trunk_input)

            # Total Loss
            loss = data_loss + config.lambda_physics * phys_loss

            loss.backward()
            optimizer.step()

            train_total_loss += loss.item()
            train_data_loss += data_loss.item()
            train_phys_loss += phys_loss.item()

        scheduler.step()

        train_total_loss /= len(train_loader)
        train_data_loss /= len(train_loader)
        train_phys_loss /= len(train_loader)

        model.eval()

        val_mse = 0
        val_rel_l2 = 0

        with torch.no_grad():

            for branch_input, trunk_input, target in val_loader:

                branch_input = branch_input.to(device)
                trunk_input = trunk_input.to(device)
                target = target.to(device)

                pred = model(branch_input, trunk_input)

                val_mse += F.mse_loss(pred, target).item()
                val_rel_l2 += relative_l2(pred, target).item()

        val_mse /= len(val_loader)
        val_rel_l2 /= len(val_loader)


        wandb.log({
            "Epoch": epoch + 1,
            "Train Total Loss": train_total_loss,
            "Train Data Loss": train_data_loss,
            "Train Physics Loss": train_phys_loss,
            "Validation MSE": val_mse,
            "Validation Rel-L2": val_rel_l2,
            "Learning Rate": scheduler.get_last_lr()[0]
        })

        print(
            f"Epoch {epoch+1:03d} | "
            f"Total {train_total_loss:.6f} | "
            f"Data {train_data_loss:.6f} | "
            f"Phys {train_phys_loss:.6f} | "
            f"Val MSE {val_mse:.6f} | "
            f"Val RelL2 {val_rel_l2:.6f}"
        )

    wandb.finish()


Training PI-DeepONet for: harmonics


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 001 | Total 0.998159 | Data 0.997575 | Phys 0.005840 | Val MSE 1.039090 | Val RelL2 0.935307
Epoch 002 | Total 0.987850 | Data 0.987037 | Phys 0.008130 | Val MSE 1.081100 | Val RelL2 1.397147
Epoch 003 | Total 0.991218 | Data 0.990694 | Phys 0.005242 | Val MSE 1.028552 | Val RelL2 0.943180
Epoch 004 | Total 0.986976 | Data 0.986688 | Phys 0.002877 | Val MSE 1.023481 | Val RelL2 0.959440
Epoch 005 | Total 0.985085 | Data 0.984802 | Phys 0.002826 | Val MSE 1.020943 | Val RelL2 0.971578
Epoch 006 | Total 0.981261 | Data 0.980775 | Phys 0.004866 | Val MSE 1.037353 | Val RelL2 0.944944
Epoch 007 | Total 0.979718 | Data 0.979422 | Phys 0.002969 | Val MSE 1.021397 | Val RelL2 0.987840
Epoch 008 | Total 0.980217 | Data 0.979793 | Phys 0.004236 | Val MSE 1.025970 | Val RelL2 0.929269
Epoch 009 | Total 0.980299 | Data 0.979907 | Phys 0.003917 | Val MSE 1.024292 | Val RelL2 0.995339
Epoch 010 | Total 0.982484 | Data 0.981881 | Phys 0.006036 | Val MSE 1.024000 | Val RelL2 0.980354
Epoch 011 

Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇████
Learning Rate,█████████▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
Train Data Loss,█▅▆▂▂▂▃▃▃▂▃▂▂▂▄▃▂▂▁▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁
Train Physics Loss,█▄▁▄▁▂▄▄▂▃▃▄▆▃▄▅▄▄▅▅▄▅▄▅▂▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆
Train Total Loss,▆█▆▅▃▄▃▄▄▄▂▅▄▃▅▃▂▃▂▃▂▂▂▁▁▂▁▂▂▂▂▁▂▂▁▁▁▂▂▁
Validation MSE,▃▃▂▂▇▂▅▄▂▂▂█▃▂▆▂▃▅▁▄▃▃▂▄▄▄▂▄▃▂▃▆▂▃▅▄▄▃▂▄
Validation Rel-L2,▁█▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Epoch,100
Learning Rate,0
Train Data Loss,0.97561
Train Physics Loss,0.00635



Training PI-DeepONet for: nurbs


Epoch 001 | Total 1.009553 | Data 1.008950 | Phys 0.006028 | Val MSE 1.105199 | Val RelL2 0.986411
Epoch 002 | Total 0.978232 | Data 0.977930 | Phys 0.003020 | Val MSE 1.106903 | Val RelL2 0.982599
Epoch 003 | Total 0.988725 | Data 0.988332 | Phys 0.003931 | Val MSE 1.106985 | Val RelL2 1.007944
Epoch 004 | Total 0.976888 | Data 0.976684 | Phys 0.002039 | Val MSE 1.112001 | Val RelL2 0.974906
Epoch 005 | Total 0.974347 | Data 0.974149 | Phys 0.001980 | Val MSE 1.133023 | Val RelL2 1.142065
Epoch 006 | Total 0.973903 | Data 0.973684 | Phys 0.002190 | Val MSE 1.105091 | Val RelL2 1.039412
Epoch 007 | Total 0.972778 | Data 0.972608 | Phys 0.001700 | Val MSE 1.103324 | Val RelL2 0.972862
Epoch 008 | Total 0.971359 | Data 0.971163 | Phys 0.001953 | Val MSE 1.116727 | Val RelL2 1.075994
Epoch 009 | Total 0.970531 | Data 0.970303 | Phys 0.002276 | Val MSE 1.119445 | Val RelL2 1.112587
Epoch 010 | Total 0.974037 | Data 0.973860 | Phys 0.001771 | Val MSE 1.114929 | Val RelL2 0.987463
Epoch 011 

Epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
Learning Rate,██████████▇▇▇▇▇▇▇▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁
Train Data Loss,█▃▅▂▂▃▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▁▁▂▂▁▂▁▁▁▁▂▂▁▁▂▁▂▂
Train Physics Loss,▅▇▂▂▁▃▁▅▅▁▃▄▃▄▂▂▂▄▁▃▂▄▃█▂▄▄▄▃▆▅▄▄▅▅▅▅▆▆▆
Train Total Loss,▇▇▇█▄▄▅▆▅▃▅▃▄▄▅▄▄▄▅▆▅▄▄▄▃▃▄▂▃▂▃▅▄▃▄▁▂▂▁▃
Validation MSE,▃▃▄█▃▂▁▃▄▄▄▂▅▄▃▅▃▅▅▅▄▄▄▃▄▄▅▅▅▅▄▅▄▅▄▄▅▅▅▄
Validation Rel-L2,▃▇▃▁▄▄▅▅▆▇▃▅▅▅▅▅█▆▄▄▅▅▅▄▆▅▅▆▆▅▅▅▆▆▆▆▆▆▆▆
Epoch,100
Learning Rate,0
Train Data Loss,0.96723
Train Physics Loss,0.00337



Training PI-DeepONet for: skelneton


Epoch 001 | Total 0.934424 | Data 0.934300 | Phys 0.001232 | Val MSE 1.280996 | Val RelL2 1.952239
Epoch 002 | Total 0.932643 | Data 0.932441 | Phys 0.002021 | Val MSE 1.311735 | Val RelL2 2.818001
Epoch 003 | Total 0.936530 | Data 0.936413 | Phys 0.001165 | Val MSE 1.257864 | Val RelL2 1.102297
Epoch 004 | Total 0.939210 | Data 0.939173 | Phys 0.000364 | Val MSE 1.264428 | Val RelL2 1.120088
Epoch 005 | Total 0.928458 | Data 0.928435 | Phys 0.000232 | Val MSE 1.285446 | Val RelL2 1.356149
Epoch 006 | Total 0.934030 | Data 0.933993 | Phys 0.000369 | Val MSE 1.276500 | Val RelL2 1.639723
Epoch 007 | Total 0.924653 | Data 0.924633 | Phys 0.000201 | Val MSE 1.297670 | Val RelL2 0.948797
Epoch 008 | Total 0.936254 | Data 0.936231 | Phys 0.000231 | Val MSE 1.285038 | Val RelL2 1.159131
Epoch 009 | Total 0.932881 | Data 0.932844 | Phys 0.000371 | Val MSE 1.267974 | Val RelL2 1.641811
Epoch 010 | Total 0.931538 | Data 0.931511 | Phys 0.000270 | Val MSE 1.288120 | Val RelL2 1.447126
Epoch 011 

Epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
Learning Rate,███████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
Train Data Loss,█▄▇▆█▆▄█▄▆▂▆▇▅▄▆▅▂▃▄▂▄▄▆▂▃▆▅▄▁▇▃▄▃▃▂▆▁▄▆
Train Physics Loss,█▂▂▃▅▃▃▃▂▁▁▁▁▁▁▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▂▂▂▂▁▂▂▂▂▂
Train Total Loss,▇▆█▇█▇▆▅▅█▅▇▆▄▇▅▆▃▆▃▁▇▃▃▆▆▇▄█▅▅▂▃▃▄▅▃▁▄▇
Validation MSE,▄▂▄▆▅▆▃▁▃▅▇▆▆▅▅█▂▄▅▄▄▄▇▄▂▅▇▃▃▅▃▇▇▁▃▄▅▅▇▅
Validation Rel-L2,▅█▂▃▃▂▁▂▄▂▃▁▁▁▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
Epoch,100
Learning Rate,0
Train Data Loss,0.91638
Train Physics Loss,0.00024
